# Module 05 — Shuffle & Sort

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import begin_module
from shared import config

spark = begin_module('module-05-shuffle-sort')
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())

## Join Strategies

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50MB')
trips.join(zones.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID').explain('formatted')

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
trips.join(zones.withColumnRenamed('LocationID', 'PULocationID'), 'PULocationID').explain('formatted')

## Storage Partitioned Join (SPJ)

In [ ]:
spark.sql('''
    SET spark.sql.sources.v2.bucketing.enabled=true;
    SET spark.sql.sources.v2.bucketing.pushPartValues.enabled=true;
    SET spark.sql.iceberg.planning.preserve-data-grouping=true;
    SET spark.sql.requireAllClusterKeysForCoPartition=false;
    SET spark.sql.autoBroadcastJoinThreshold=-1;
''')

In [ ]:
trips_a = trips.select('VendorID', 'PULocationID', 'total_amount', 'tpep_pickup_datetime')
trips_b = trips.select('VendorID', 'DOLocationID', 'total_amount', 'tpep_dropoff_datetime')
trips_a.writeTo('local.default.trips_a').using('iceberg').partitionedBy('PULocationID').createOrReplace()
trips_b.writeTo('local.default.trips_b').using('iceberg').partitionedBy('PULocationID').createOrReplace()

In [ ]:
spj = spark.sql('''SELECT a.VendorID, a.total_amount, b.total_amount
    FROM local.default.trips_a a JOIN local.default.trips_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
spj.explain('formatted')
spj.limit(10).show()

## Table Ordering

In [ ]:
spark.sql('CREATE OR REPLACE TABLE local.default.trips_ordered USING iceberg AS SELECT * FROM local.default.trips_a')
spark.sql('ALTER TABLE local.default.trips_ordered WRITE ORDERED BY (PULocationID, total_amount)')
spark.sql("SHOW TBLPROPERTIES local.default.trips_ordered").filter("key LIKE 'write.%' OR key = 'sort-order'").show(truncate=False)

## Key Takeaways

1. Broadcast avoids shuffle for small tables.
2. SPJ eliminates shuffle when Iceberg partitions align.
3. Table sort order improves reads but adds write cost.